# Length of Stay - Charlson

In [ ]:
import yaml
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

# Load all models
ALL_MODELS = yaml.safe_load(open("../config.yml"))["models"]

relevant_key = "charlson_los"

## Index-Specific Train/Test Splits

Create views of the train/test splits which have the comorbidities for each index present.

### Data Aggregation

The training data can be quite large which results in extremely large inputs for the models. For example, the MACSS has 100 comorbidities so the training data is a N x 100 matrix where N is the number of rows.

To improve the training time for our models, we can 'compress' the data and represent it by identifying each unique combination of comorbidities and the number of times it occurred.

For example, given the following row-level data:

| COMORB_1 | COMORB_2 | COMORB_3 |
| -------- | -------- | -------- |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    0     |
|    1     |    0     |    1     |
|    1     |    0     |    1     |

We would represent it as:

| COMORB_1 | COMORB_2 | COMORB_3 | N |
| -------- | -------- | -------- | - |
|    1     |    1     |    1     | 7 |
|    1     |    1     |    0     | 1 |
|    1     |    0     |    1     | 2 |

In [ ]:

MODEL_CONFIG = ALL_MODELS[relevant_key]
print(f"\nProcessing {relevant_key} (Target: {MODEL_CONFIG['class']})...")

In [ ]:
# Load Data
charls_los_training = pd.read_csv(f"../datasets/{MODEL_CONFIG['dataset_training']}")
charls_los_testing = pd.read_csv(f"../datasets/{MODEL_CONFIG['dataset_testing']}")

In [ ]:
# Data Aggregation
charls_los_training_agg = charls_los_training.groupby(list(charls_los_training.columns),dropna=False).size().reset_index(name='N')
charls_los_testing_agg = charls_los_testing.groupby(list(charls_los_testing.columns),dropna=False).size().reset_index(name='N')


In [ ]:
# Define input columns
input_cols = [i for i in charls_los_training_agg.columns if "C_" in i]

In [ ]:
# Train
clf = LinearRegression()
clf.fit(charls_los_training_agg[input_cols], charls_los_training_agg[MODEL_CONFIG["class"]], sample_weight=charls_los_training_agg['N'])

In [ ]:
# Predict
y_train_pred = clf.predict(charls_los_training_agg[input_cols])
y_train_true = charls_los_training_agg[MODEL_CONFIG["class"]]

In [ ]:
y_test_pred = clf.predict(charls_los_testing_agg[input_cols])
y_test_true = charls_los_testing_agg[MODEL_CONFIG["class"]]

In [ ]:
# Metrics
mse_train = mean_squared_error(y_train_true, y_train_pred, sample_weight=charls_los_training_agg['N'])
mae_train = mean_absolute_error(y_train_true, y_train_pred, sample_weight=charls_los_training_agg['N'])
r2_train = r2_score(y_train_true, y_train_pred, sample_weight=charls_los_training_agg['N'])

mse_test = mean_squared_error(y_test_true, y_test_pred, sample_weight=charls_los_testing_agg['N'])
mae_test = mean_absolute_error(y_test_true, y_test_pred, sample_weight=charls_los_testing_agg['N'])
r2_test = r2_score(y_test_true, y_test_pred, sample_weight=charls_los_testing_agg['N'])

print(f"  Training R2: {r2_train:.4f}, MSE: {mse_train:.4f}")
print(f"  Test R2: {r2_test:.4f}, MSE: {mse_test:.4f}")


In [ ]:
# Save
if not os.path.exists('../models'):
    os.makedirs('../models')

save_path = f'../models/los_charlson_{MODEL_CONFIG["class"]}.joblib'
joblib.dump(clf, save_path)
print(f"  Model saved to {save_path}")